# Evaluation-Only Notebook: NL → Java → C# (Base vs. Fine-tuned)

**Purpose:** This notebook is **evaluation-only** — it contains **no training, no LoRA
fine-tuning, no `Trainer`/`SFTTrainer`, no optimizer/scheduler, no checkpoint saving, and no
`push_to_hub` calls.** It only *loads* already-published Hugging Face models and benchmarks
them.

**Models evaluated (loaded directly, already merged — NOT LoRA adapters):**
| Stage | Fine-tuned | Base |
|---|---|---|
| 1: NL → Java | `shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5` | `Qwen/Qwen2.5-Coder-1.5B-Instruct` |
| 2: Java → C# | `shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5` | `Qwen/Qwen2.5-Coder-1.5B-Instruct` |



In [1]:
%%capture --no-stderr
!pip install -q -U unsloth
!pip install -q -U sacrebleu codebleu code-bert-score tree-sitter-java tree-sitter-c-sharp "datasets>=2.14" javalang
!pip install -q -U "tree-sitter>=0.24"

In [2]:
import gc
import json
import logging
import math
import os
import platform
import random
import re
import shutil
import subprocess
import tarfile
import tempfile
import time
import urllib.request
import csv
from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Sequence

import numpy as np
import torch
from datasets import load_dataset
print("Core imports ready | torch", torch.__version__,
      "| CUDA available:", torch.cuda.is_available())

Core imports ready | torch 2.10.0+cu128 | CUDA available: True


## 3. Configuration

In [3]:
@dataclass(frozen=True)
class Config:
    """Single source of truth for every tunable value in this evaluation-only notebook."""

    SEED: int = 3407

    BASE_MODEL_NAME: str = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
    MAX_LENGTH: int = 2048

    HF_REPO: str = "shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5"
    HF_REPO_STAGE2: str = "shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5"

    DATASET_NAME: str = "code_search_net"
    DATASET_CONFIG: str = "java"
    NUM_STAGE1_SAMPLES: int = 100

    STAGE2_MAX_SEQ_LEN: int = 1024
    STAGE2_MAX_SAMPLES: int = 6000
    NUM_STAGE2_SAMPLES: int = 50


CFG = Config()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Configuration loaded. DEVICE={DEVICE} | run timestamp={TIMESTAMP}")

Configuration loaded. DEVICE=cuda | run timestamp=20260801_034637


In [4]:
def configure_logging(name: str = "EvalOnlyNotebook", level: int = logging.INFO) -> logging.Logger:
    """Configure a structured root logger and return a named child logger."""
    logging.basicConfig(
        level=level,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
        force=True,
    )
    return logging.getLogger(name)


logger = configure_logging()


def set_global_seed(seed: int) -> None:
    """Seed Python, NumPy, PyTorch (and HF) RNGs for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        from transformers import set_seed as hf_set_seed
        hf_set_seed(seed)
    except Exception:
        pass
    logger.info("Global seed set to %d", seed)


def log_gpu_info(tag: str = "") -> None:
    """Log GPU name + current memory usage, or warn if running on CPU."""
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        logger.info(
            "GPU [%s] %s | %.2f GB allocated, %.2f GB reserved, %.1f GB total",
            tag, name, alloc, reserved, total,
        )
    else:
        logger.warning(
            "CUDA not available - running on CPU (evaluation will be very slow).")


def free_memory(*names: str, scope: Optional[dict] = None) -> None:
    """Delete named globals, run GC, and empty the CUDA cache."""
    scope = scope if scope is not None else globals()
    for n in names:
        if n in scope:
            del scope[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    log_gpu_info("after free_memory")


set_global_seed(CFG.SEED)
log_gpu_info("startup")

03:46:54 | WARNING | torchao | Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
03:46:56 | INFO | EvalOnlyNotebook | Global seed set to 3407
03:46:56 | INFO | EvalOnlyNotebook | GPU [startup] Tesla T4 | 0.00 GB allocated, 0.00 GB reserved, 15.6 GB total


In [5]:
from unsloth import FastLanguageModel


def load_model_and_tokenizer(model_name: str, max_seq_length: int, load_in_4bit: bool = True):
    """Load an already-merged causal LM + tokenizer via Unsloth (4-bit, fast inference)."""
    try:
        logger.info("Loading model '%s' (max_seq_length=%d, 4bit=%s) ...",
                    model_name, max_seq_length, load_in_4bit)
        mdl, tok = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=max_seq_length,
            load_in_4bit=load_in_4bit,
        )
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        logger.info("Model '%s' loaded. EOS token id=%s",
                    model_name, tok.eos_token_id)
        log_gpu_info(f"after loading {model_name}")
        return mdl, tok
    except Exception:
        logger.exception("Failed to load model '%s'.", model_name)
        raise

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
stage1_model, stage1_tokenizer = load_model_and_tokenizer(
    CFG.HF_REPO, CFG.MAX_LENGTH)

03:47:10 | INFO | EvalOnlyNotebook | Loading model 'shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5' (max_seq_length=2048, 4bit=True) ...
03:47:10 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:47:10 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5/083cedb2462d20e422b14981799f549c4662acfe/config.json "HTTP/1.1 200 OK"
03:47:10 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5/083cedb2462d20e422b14981799f549c4662acfe/config.json "HTTP/1.1 200 OK"
03:47:10 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


==((====))==  Unsloth 2026.7.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


03:47:11 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/revision/main "HTTP/1.1 200 OK"
03:47:11 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.gitattributes "HTTP/1.1 307 Temporary Redirect"
03:47:11 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.gitattributes "HTTP/1.1 200 OK"
03:47:11 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.gitattributes "HTTP/1.1 200 OK"
03:47:11 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.1 307 Temporary Redirect"
03:47:11 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/model.saf

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

03:47:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
03:47:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5/083cedb2462d20e422b14981799f549c4662acfe/generation_config.json "HTTP/1.1 200 OK"
03:47:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:47:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5/083cedb2462d20e422b14981799f549c4662acfe/config.json "HTTP/1.1 200 OK"
03:47:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_V5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
03:47:34 | INFO | httpx 

## 6. Load Stage 2 Model (Fine-tuned Java → C#)

In [7]:
stage2_model, stage2_tokenizer = load_model_and_tokenizer(
    CFG.HF_REPO_STAGE2, CFG.STAGE2_MAX_SEQ_LEN)

03:47:38 | INFO | EvalOnlyNotebook | Loading model 'shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5' (max_seq_length=1024, 4bit=True) ...
03:47:38 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:47:38 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5/d8415129d7568c573131e7f7b82ef509ed2fb4a2/config.json "HTTP/1.1 200 OK"
03:47:38 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5/d8415129d7568c573131e7f7b82ef509ed2fb4a2/config.json "HTTP/1.1 200 OK"
03:47:38 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


==((====))==  Unsloth 2026.7.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


03:47:38 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/revision/main "HTTP/1.1 200 OK"
03:47:38 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.1 307 Temporary Redirect"
03:47:38 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.1 200 OK"
03:47:38 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.gitattributes "HTTP/1.1 307 Temporary Redirect"
03:47:38 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.1 200 OK"
03:47:38 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10e

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

03:48:09 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
03:48:09 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5/d8415129d7568c573131e7f7b82ef509ed2fb4a2/generation_config.json "HTTP/1.1 200 OK"
03:48:10 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:48:10 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5/d8415129d7568c573131e7f7b82ef509ed2fb4a2/config.json "HTTP/1.1 200 OK"
03:48:10 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
03:4

In [8]:
base_model, base_tokenizer = load_model_and_tokenizer(
    CFG.BASE_MODEL_NAME, max(CFG.MAX_LENGTH, CFG.STAGE2_MAX_SEQ_LEN))

03:48:14 | INFO | EvalOnlyNotebook | Loading model 'Qwen/Qwen2.5-Coder-1.5B-Instruct' (max_seq_length=2048, 4bit=True) ...
03:48:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/qwen2.5-coder-1.5b-instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:48:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:48:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
03:48:14 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
03:48:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/qwen2.5-coder-1.5b-instruct-bnb-4bit/res

==((====))==  Unsloth 2026.7.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


03:48:14 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/revision/main "HTTP/1.1 200 OK"
03:48:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 307 Temporary Redirect"
03:48:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 200 OK"
03:48:14 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 200 OK"
03:48:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.gitattributes "HTTP/1.1 307 Temporary Redirect"
03:48:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.git

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

03:48:26 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/qwen2.5-coder-1.5b-instruct-bnb-4bit/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
03:48:26 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
03:48:26 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/generation_config.json "HTTP/1.1 200 OK"
03:48:26 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/qwen2.5-coder-1.5b-instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:48:26 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:48:26 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/ap

## Load Datasets

In [9]:
STAGE1_DOC_COL = "func_documentation_string"
STAGE1_CODE_COL = "func_code_string"


def load_code_dataset(split: str, dataset_name: str = CFG.DATASET_NAME,
                      config: str = CFG.DATASET_CONFIG):
    """Load a split of the code dataset with logging and error handling."""
    try:
        logger.info("Loading %s/%s split='%s' ...",
                    dataset_name, config, split)
        ds = load_dataset(dataset_name, config, split=split)
        logger.info("Loaded %d examples for split '%s'.", len(ds), split)
        return ds
    except Exception:
        logger.exception("Failed to load dataset %s/%s (split=%s).",
                         dataset_name, config, split)
        raise


def validate_dataset(ds, required_columns: Sequence[str], n_check: int = 100) -> None:
    """Validate that required columns exist and have non-empty values."""
    if ds is None or len(ds) == 0:
        raise ValueError("Dataset is empty or None.")
    missing = [c for c in required_columns if c not in ds.column_names]
    if missing:
        raise ValueError(f"Dataset missing required columns: {missing} "
                         f"(available: {ds.column_names})")
    sample = ds.select(range(min(n_check, len(ds))))
    for col in required_columns:
        n_null = sum(1 for v in sample[col] if v is None or not str(v).strip())
        if n_null:
            logger.warning("Column '%s' has %d empty values in first %d rows.",
                           col, n_null, len(sample))
    logger.info("Dataset validation passed for columns: %s",
                list(required_columns))


test_data = load_code_dataset("test")
validate_dataset(test_data, [STAGE1_DOC_COL, STAGE1_CODE_COL])
test_subset = test_data.select(
    range(min(CFG.NUM_STAGE1_SAMPLES, len(test_data))))
logger.info("Stage 1 evaluation subset: %d examples.", len(test_subset))

03:48:30 | INFO | EvalOnlyNotebook | Loading code_search_net/java split='test' ...
03:48:30 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code_search_net/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
03:48:30 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code-search-net/code_search_net/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
03:48:30 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/code-search-net/code_search_net/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/README.md "HTTP/1.1 200 OK"
03:48:30 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/code-search-net/code_search_net/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

03:48:30 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/code_search_net.py "HTTP/1.1 307 Temporary Redirect"
03:48:30 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code-search-net/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/code_search_net.py "HTTP/1.1 404 Not Found"
03:48:30 | INFO | httpx | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/code_search_net/code_search_net.py "HTTP/1.1 404 Not Found"
03:48:30 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/code_search_net/revision/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555 "HTTP/1.1 307 Temporary Redirect"
03:48:30 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/code-search-net/code_search_net/revision/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555 "HTTP/1.1 200 OK"
03:48:30 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/dat

java/train-00000-of-00001.parquet:   0%|          | 0.00/390M [00:00<?, ?B/s]

03:48:35 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/java/test-00000-of-00001.parquet "HTTP/1.1 307 Temporary Redirect"
03:48:35 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code-search-net/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/java/test-00000-of-00001.parquet "HTTP/1.1 302 Found"


...

java/validation-00000-of-00001.parquet:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/454451 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/26909 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15328 [00:00<?, ? examples/s]

03:48:41 | INFO | EvalOnlyNotebook | Loaded 26909 examples for split 'test'.
03:48:41 | INFO | EvalOnlyNotebook | Dataset validation passed for columns: ['func_documentation_string', 'func_code_string']
03:48:41 | INFO | EvalOnlyNotebook | Stage 1 evaluation subset: 100 examples.


In [10]:
DATASET_CANDIDATES = [   
    ("CodeXGLUE-Java-CS", dict(path="google/code_x_glue_cc_code_to_code_trans"), "java", "cs"),
]


def _normalize_split(ds, java_col: Optional[str], csharp_col: Optional[str]):
    """Return a list of {'java', 'cs'} dicts, or None if columns are not usable."""
    cols = set(ds.column_names)
    if java_col is None or java_col not in cols:
        java_col = next((c for c in cols if c.lower() in (
            "java", "java_code", "src", "source")), None)
    if csharp_col is None or csharp_col not in cols:
        csharp_col = next((c for c in cols if c.lower() in (
            "cs", "csharp", "c#", "cs_code", "tgt", "target")), None)
    if not java_col or not csharp_col:
        return None
    pairs = [{"java": j, "cs": c}
             for j, c in zip(ds[java_col], ds[csharp_col])
             if isinstance(j, str) and isinstance(c, str) and j.strip() and c.strip()]
    return pairs or None


def load_java_csharp_dataset():
    """Try each candidate dataset in order; return (label, {split: pairs})."""
    for label, kwargs, java_col, csharp_col in DATASET_CANDIDATES:
        try:
            logger.info("Trying Java/C# dataset: %s (%s) ...",
                        label, kwargs.get("path"))
            raw = load_dataset(**kwargs)
            split_names = list(raw.keys()) if hasattr(
                raw, "keys") else ["train"]
            collected = {}
            for split in split_names:
                pairs = _normalize_split(raw[split], java_col, csharp_col)
                if pairs:
                    collected[split] = pairs
            if collected:
                total = sum(len(v) for v in collected.values())
                logger.info("Loaded %s: %d Java/C# pairs across splits %s",
                            label, total, list(collected))
                return label, collected
            logger.warning(
                "%s has no usable Java/C# columns, skipping.", label)
        except Exception as exc:
            logger.warning("%s unavailable (%s: %s). Falling back.",
                           label, type(exc).__name__, exc)
    raise RuntimeError(
        "No Java<->C# dataset could be loaded from any candidate.")


DATASET_NAME_STAGE2, raw_pairs = load_java_csharp_dataset()
logger.info("Selected Stage 2 dataset: %s", DATASET_NAME_STAGE2)

03:48:41 | INFO | EvalOnlyNotebook | Trying Java/C# dataset: XLCoST (codeparrot/xlcost-text-to-code) ...
03:48:41 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/codeparrot/xlcost-text-to-code/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
03:48:41 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/codeparrot/xlcost-text-to-code/60c5c133f043a5cffe162f9de1c62b9d88f309cf/README.md "HTTP/1.1 200 OK"
03:48:41 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/codeparrot/xlcost-text-to-code/60c5c133f043a5cffe162f9de1c62b9d88f309cf/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

03:48:42 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/codeparrot/xlcost-text-to-code/resolve/60c5c133f043a5cffe162f9de1c62b9d88f309cf/xlcost-text-to-code.py "HTTP/1.1 307 Temporary Redirect"
03:48:42 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/codeparrot/xlcost-text-to-code/60c5c133f043a5cffe162f9de1c62b9d88f309cf/xlcost-text-to-code.py "HTTP/1.1 200 OK"
03:48:42 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/codeparrot/xlcost-text-to-code/60c5c133f043a5cffe162f9de1c62b9d88f309cf/xlcost-text-to-code.py "HTTP/1.1 200 OK"


xlcost-text-to-code.py: 0.00B [00:00, ?B/s]

03:48:42 | WARNING | EvalOnlyNotebook | XLCoST unavailable (RuntimeError: Dataset scripts are no longer supported, but found xlcost-text-to-code.py). Falling back.
03:48:42 | INFO | EvalOnlyNotebook | Trying Java/C# dataset: CodeTransOcean (WeixiangYan/CodeTransOcean) ...
03:48:42 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/WeixiangYan/CodeTransOcean/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
03:48:42 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/WeixiangYan/CodeTransOcean/8985d0851100f08e94735b2f2556a795ba8c69bb/README.md "HTTP/1.1 200 OK"
03:48:42 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/WeixiangYan/CodeTransOcean/8985d0851100f08e94735b2f2556a795ba8c69bb/README.md "HTTP/1.1 200 OK"


...

data/test-00000-of-00001.parquet:   0%|          | 0.00/170k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10300 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

03:48:45 | INFO | EvalOnlyNotebook | Loaded CodeXGLUE-Java-CS: 11800 Java/C# pairs across splits ['train', 'validation', 'test']
03:48:45 | INFO | EvalOnlyNotebook | Selected Stage 2 dataset: CodeXGLUE-Java-CS


In [11]:
random.seed(CFG.SEED)

all_pairs = []
for split_pairs in raw_pairs.values():
    all_pairs.extend(split_pairs)
logger.info("Raw Java/C# pairs: %d", len(all_pairs))


def is_valid_pair(java_code: str, csharp_code: str) -> bool:
    j, c = java_code.strip(), csharp_code.strip()
    if not j or not c:
        return False
    if len(j) < 10 or len(c) < 10:
        return False
    if not any(tok in j for tok in (";", "{", "}")):
        return False
    if not any(tok in c for tok in (";", "{", "}")):
        return False
    return True


seen = set()
clean_pairs = []
for p in all_pairs:
    if not is_valid_pair(p["java"], p["cs"]):
        continue
    key = (p["java"].strip(), p["cs"].strip())
    if key in seen:
        continue
    seen.add(key)
    clean_pairs.append({"java": p["java"].strip(), "cs": p["cs"].strip()})

random.shuffle(clean_pairs)
logger.info("Clean, de-duplicated pairs: %d", len(clean_pairs))
clean_pairs = clean_pairs[:CFG.STAGE2_MAX_SAMPLES]

if len(clean_pairs) < 20:
    raise ValueError(
        f"Too few valid Java/C# pairs ({len(clean_pairs)}) to evaluate Stage 2. "
        "Check the dataset loader / cleaning filters.")

val_size = min(max(50, int(0.05 * len(clean_pairs))),
               max(1, len(clean_pairs) // 5))
val_pairs = clean_pairs[:val_size]
logger.info("Stage 2 held-out validation slice: %d pairs.", len(val_pairs))

eval_pairs = val_pairs[:min(CFG.NUM_STAGE2_SAMPLES, len(val_pairs))]
logger.info("Stage 2 evaluation subset: %d examples.", len(eval_pairs))

03:48:45 | INFO | EvalOnlyNotebook | Raw Java/C# pairs: 11800
03:48:45 | INFO | EvalOnlyNotebook | Clean, de-duplicated pairs: 11800
03:48:45 | INFO | EvalOnlyNotebook | Stage 2 held-out validation slice: 300 pairs.
03:48:45 | INFO | EvalOnlyNotebook | Stage 2 evaluation subset: 50 examples.


## Compilation Utilities

In [12]:
MARKDOWN_FENCE_RE = re.compile(
    r"```[ \t]*[a-zA-Z0-9_+-]*\r?\n(.*?)(?:```|\Z)", re.DOTALL)
JAVA_TYPE_DECL_RE = re.compile(
    r"\b(?:public\s+)?(?:final\s+)?(?:abstract\s+)?(class|interface|enum|record)\s+(\w+)")
JAVA_HEADER_LINE_RE = re.compile(
    r"^\s*(?:package\s+[\w.]+\s*;|import\s+(?:static\s+)?[\w.*]+\s*;)\s*$")
JAVA_COMMENT_RE = re.compile(r"//[^\n]*|/\*.*?\*/", re.DOTALL)
JAVA_STRING_RE = re.compile(r'"(?:\\.|[^"\\])*"')
CSHARP_TYPE_DECL_RE = re.compile(
    r"\b(class|struct|interface|enum|namespace|record)\b")


def _extract_code_from_markdown(text: str) -> str:   
    match = MARKDOWN_FENCE_RE.search(text)
    return match.group(1).strip() if match else text.strip()


def _strip_code_noise(code: str) -> str:  
    return JAVA_COMMENT_RE.sub("", JAVA_STRING_RE.sub('""', code))


def _prepare_java_source(code: str) -> "tuple[str, str]":   
    code = _extract_code_from_markdown(code)
    match = JAVA_TYPE_DECL_RE.search(_strip_code_noise(code))
    if match:
        return code, match.group(2)

    lines = code.splitlines()
    header_lines = [ln for ln in lines if JAVA_HEADER_LINE_RE.match(ln)]
    body_lines = [ln for ln in lines if not JAVA_HEADER_LINE_RE.match(ln)]
    header_src = "\n".join(header_lines)
    body_src = "\n".join(body_lines)
    prefix = f"{header_src}\n" if header_src.strip() else ""
    class_name = "GeneratedProgram"
    return f"{prefix}final class {class_name} {{\n{body_src}\n}}\n", class_name


def _prepare_csharp_source(code: str) -> str:   
    code = _extract_code_from_markdown(code)
    if CSHARP_TYPE_DECL_RE.search(_strip_code_noise(code)):
        if "using System;" not in code and "using System " not in code:
            code = "using System;\n" + code
        return code
    return ("using System;\nusing System.Collections.Generic;\n\n"
            "public class GeneratedProgram\n{\n" + code + "\n}\n")

In [13]:
_JAVAC_PATH: Optional[str] = None
_DOTNET_PATH: Optional[str] = None


def _ensure_javac() -> Optional[str]:
    """Locate `javac`, or best-effort install a portable JDK. Returns the path or None."""
    global _JAVAC_PATH
    if _JAVAC_PATH:
        return _JAVAC_PATH
    found = shutil.which("javac")
    if found:
        _JAVAC_PATH = found
        return found
    try:
        logger.info(
            "javac not found - attempting `apt-get install openjdk-17-jdk-headless` ...")
        subprocess.run(["apt-get", "update", "-qq"], check=False,
                       capture_output=True, timeout=120)
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"],
                       check=False, capture_output=True, timeout=300)
        found = shutil.which("javac")
        if found:
            _JAVAC_PATH = found
            return found
    except Exception:
        logger.warning(
            "apt-get based javac install failed; trying a portable JDK download.")
    try:
        jdk_dir = Path(tempfile.gettempdir()) / "portable_jdk17"
        if not jdk_dir.exists():
            url = ("https://api.adoptium.net/v3/binary/latest/17/ga/"
                   f"{'windows' if platform.system() == 'Windows' else 'linux'}/x64/jdk/"
                   "hotspot/normal/eclipse")
            archive_path = Path(tempfile.gettempdir()) / "temurin17.tar.gz"
            urllib.request.urlretrieve(url, archive_path)
            jdk_dir.mkdir(parents=True, exist_ok=True)
            with tarfile.open(archive_path) as tf:
                tf.extractall(jdk_dir)
        for candidate in jdk_dir.rglob("javac*"):
            if candidate.is_file():
                _JAVAC_PATH = str(candidate)
                return _JAVAC_PATH
    except Exception:
        logger.exception("Portable JDK download/install failed.")
    logger.warning(
        "javac could not be installed; Java compilation checks will report False.")
    return None


def _ensure_dotnet() -> Optional[str]:
    """Locate `dotnet`, or best-effort install it via Microsoft's official script."""
    global _DOTNET_PATH
    if _DOTNET_PATH:
        return _DOTNET_PATH
    found = shutil.which("dotnet")
    if found:
        _DOTNET_PATH = found
        return found
    try:
        logger.info(
            "dotnet not found - attempting Microsoft's official install script ...")
        install_dir = Path.home() / ".dotnet"
        script_path = Path(tempfile.gettempdir()) / "dotnet-install.sh"
        urllib.request.urlretrieve(
            "https://dot.net/v1/dotnet-install.sh", script_path)
        os.chmod(script_path, 0o755)
        subprocess.run([str(script_path), "--channel", "8.0", "--install-dir", str(install_dir)],
                       check=False, capture_output=True, timeout=300)
        candidate = install_dir / "dotnet"
        if candidate.exists():
            os.environ["PATH"] = f"{install_dir}{os.pathsep}" + \
                os.environ.get("PATH", "")
            _DOTNET_PATH = str(candidate)
            return _DOTNET_PATH
    except Exception:
        logger.exception("dotnet install script failed.")
    for alt in ("csc", "mcs"):
        found = shutil.which(alt)
        if found:
            logger.info(
                "Falling back to '%s' compiler for C# validation.", alt)
            return found
    logger.warning(
        "dotnet/csc/mcs could not be installed; C# compilation checks will report False.")
    return None

In [14]:
def compile_java(code: str, timeout: int = 20) -> Dict[str, object]:
    """Attempt to compile a single Java snippet with `javac` in a throwaway temp dir."""
    javac = _ensure_javac()
    if not javac:
        return {"success": False, "error": "javac unavailable"}
    try:
        source, class_name = _prepare_java_source(code)
    except Exception as exc:
        return {"success": False, "error": f"prepare failed: {exc}"}
    with tempfile.TemporaryDirectory() as tmpdir:
        src_path = Path(tmpdir) / f"{class_name}.java"
        src_path.write_text(source, encoding="utf-8")
        try:
            result = subprocess.run(
                [javac, "-d", tmpdir, str(src_path)],
                capture_output=True, text=True, timeout=timeout,
            )
            return {"success": result.returncode == 0, "stderr": result.stderr}
        except Exception as exc:
            return {"success": False, "error": str(exc)}


def compile_csharp(code: str, timeout: int = 30) -> Dict[str, object]:
    """Attempt to compile a single C# snippet via `dotnet build` (falls back to csc/mcs)."""
    compiler = _ensure_dotnet()
    if not compiler:
        return {"success": False, "error": "no C# compiler available"}
    source = _prepare_csharp_source(code)
    with tempfile.TemporaryDirectory() as tmpdir:
        try:
            if compiler.endswith("dotnet") or Path(compiler).name == "dotnet":
                proj_dir = Path(tmpdir)
                (proj_dir / "Program.cs").write_text(source, encoding="utf-8")
                (proj_dir / "temp.csproj").write_text(
                    '<Project Sdk="Microsoft.NET.Sdk"><PropertyGroup>'
                    "<OutputType>Exe</OutputType><TargetFramework>net8.0</TargetFramework>"
                    "<Nullable>disable</Nullable><ImplicitUsings>disable</ImplicitUsings>"
                    "</PropertyGroup></Project>",
                    encoding="utf-8",
                )
                result = subprocess.run(
                    [compiler, "build", str(proj_dir), "-v", "quiet"],
                    capture_output=True, text=True, timeout=timeout,
                )
            else:
                src_path = Path(tmpdir) / "Program.cs"
                src_path.write_text(source, encoding="utf-8")
                result = subprocess.run(
                    [compiler, str(src_path)],
                    capture_output=True, text=True, timeout=timeout,
                )
            return {"success": result.returncode == 0, "stderr": result.stderr}
        except Exception as exc:
            return {"success": False, "error": str(exc)}


def compute_compilation_rate(code_snippets: List[str], language: str) -> float:
    """Compile each snippet with a real compiler and return the fraction that succeed."""
    compile_fn = compile_java if language == "java" else compile_csharp
    successes = 0
    for snippet in code_snippets:
        result = compile_fn(snippet)
        if result.get("success"):
            successes += 1
    return successes / max(1, len(code_snippets))


def is_syntactically_valid_java(code: str) -> bool:
    """Cheap, compiler-free brace/paren-balance heuristic for Java syntax validity."""
    code = _extract_code_from_markdown(code)
    return code.count("{") == code.count("}") and code.count("(") == code.count(")")


def is_syntactically_valid_csharp(code: str) -> bool:
    """Cheap, compiler-free brace/paren-balance heuristic for C# syntax validity."""
    code = _extract_code_from_markdown(code)
    return code.count("{") == code.count("}") and code.count("(") == code.count(")")


def compute_lenient_compilation_rate(code_snippets: List[str], language: str) -> float:
    """Fraction of snippets that pass the cheap brace/paren-balance heuristic (no compiler needed)."""
    check_fn = is_syntactically_valid_java if language == "java" else is_syntactically_valid_csharp
    valid = sum(1 for snippet in code_snippets if check_fn(snippet))
    return valid / max(1, len(code_snippets))

In [15]:
def _show_class_structured_sample(raw_code: str, prepare_fn, compile_fn, label: str) -> None:
    """Print the raw model output next to the CLASS-STRUCTURED (wrapped) source """
    print(f"--- {label}: raw model output ---")
    print((raw_code or "")[:500])
    try:
        prepared = prepare_fn(raw_code)
        wrapped_source = prepared[0] if isinstance(
            prepared, tuple) else prepared
    except Exception as exc:
        print(f"[class-structure wrap failed: {exc}]")
        return
    print(f"--- {label}: class-structured source sent to the compiler ---")
    print(wrapped_source[:800])
    result = compile_fn(raw_code)
    status = "SUCCESS" if result.get("success") else "FAILED"
    print(f"--- {label}: compile result = {status} ---")
    if not result.get("success"):
        print((result.get("stderr") or result.get("error") or "")[:800])
    print()


def diagnose_compilation_rate(predictions: List[str], prepare_fn, compile_fn,
                              language_label: str, rate: float, n_samples: int = 2) -> None:
    """Print `n_samples` class-structured (wrapped) code samples """
    print("=" * 70)
    print(f"DIAGNOSTIC: class-structured {language_label} output "
          f"(first {min(n_samples, len(predictions))} predictions)")
    print("=" * 70)
    for pred in predictions[:n_samples]:
        _show_class_structured_sample(
            pred, prepare_fn, compile_fn, f"{language_label} prediction")
    if rate == 0.0:
        print(f"NOTE: 0% {language_label} compilation rate detected. Check the class-structured")

## Java Compilation Failure Classification


In [16]:
JAVA_SYNTAX_ERROR_PATTERNS = [
    r"';'\s*expected",
    r"'\)'\s*expected",
    r"'\('\s*expected",
    r"'\}'\s*expected",
    r"'\{'\s*expected",
    r"illegal start of expression",
    r"illegal start of type",
    r"reached end of file while parsing",
    r"class,\s*interface,(?:\s*or)?\s*enum expected",
    r"invalid method declaration",
    r"identifier expected",
    r"not a statement",
]

JAVA_TYPE_MISMATCH_PATTERNS = [
    r"incompatible types",
    r"bad operand types",
    r"cannot convert from",
]

JAVA_API_MISMATCH_PATTERNS = [
    r"method\s+\S+.*cannot be applied to given types",
    r"no suitable method found",
    r"constructor\s+\S+.*cannot be applied",
]

JAVA_MISSING_DEPENDENCY_PATTERNS = [
    r"cannot find symbol",
    r"package\s+[\w.]+\s+does not exist",
    r"class\s+\w+\s+not found",
]

LENIENT_PASS_CATEGORIES = {"MISSING_DEPENDENCY"}


def _matches_any(patterns: List[str], text: str) -> bool:
    """True if `text` matches any regex in `patterns` (case-insensitive)."""
    return any(re.search(pattern, text, re.IGNORECASE) for pattern in patterns)


def classify_javac_stderr(stderr: str) -> str:
    """Classify a `javac` failure's raw stderr into one failure category.

    Checked in order of severity: SYNTAX_ERROR > TYPE_MISMATCH > API_MISMATCH >
    MISSING_DEPENDENCY > OTHER. Returns one of those 5 strings - callers should only invoke
    this once compilation has already failed (a successful compile is reported as "PASS" by
    `analyze_java_compilation`, not by this function).
    """
    text = stderr or ""
    if not text.strip():
        return "OTHER"
    if _matches_any(JAVA_SYNTAX_ERROR_PATTERNS, text):
        return "SYNTAX_ERROR"
    if _matches_any(JAVA_TYPE_MISMATCH_PATTERNS, text):
        return "TYPE_MISMATCH"
    if _matches_any(JAVA_API_MISMATCH_PATTERNS, text):
        return "API_MISMATCH"
    if _matches_any(JAVA_MISSING_DEPENDENCY_PATTERNS, text):
        return "MISSING_DEPENDENCY"
    return "OTHER"


def analyze_java_compilation(code: str, timeout: int = 20) -> Dict[str, object]:
    """Compile one Java snippet and return a structured classification result:
    `{"status", "category", "compiler_output", "is_strict_compile_success",
    "is_lenient_compile_success"}`. Reuses `compile_java` (Section 9) so the actual compile
    step - class-wrapping, javac invocation, temp-dir handling - stays a single implementation.
    """
    result = compile_java(code, timeout=timeout)
    is_success = bool(result.get("success"))
    compiler_output = result.get("stderr") or result.get("error") or ""
    category = "PASS" if is_success else classify_javac_stderr(compiler_output)
    is_lenient_success = is_success or category in LENIENT_PASS_CATEGORIES
    return {
        "status": "PASS" if is_success else "FAILED",
        "category": category,
        "compiler_output": compiler_output,
        "is_strict_compile_success": is_success,
        "is_lenient_compile_success": is_lenient_success,
    }


def compute_java_compilation_report(code_snippets: List[str], timeout: int = 20) -> Dict[str, object]:
    """Run `analyze_java_compilation` over every snippet and aggregate the results into
    per-category counts plus the strict and lenient compilation rates."""
    results = [analyze_java_compilation(snippet, timeout=timeout)
               for snippet in code_snippets]
    counts = Counter(r["category"] for r in results)
    n = max(1, len(results))
    strict_rate = sum(
        1 for r in results if r["is_strict_compile_success"]) / n
    lenient_rate = sum(
        1 for r in results if r["is_lenient_compile_success"]) / n
    return {
        "results": results,
        "counts": counts,
        "strict_rate": strict_rate,
        "lenient_rate": lenient_rate,
    }


def print_compilation_summary(report: Dict[str, object], title: str = "Compilation Summary") -> None:
    """Print the category breakdown (PASS/SYNTAX_ERROR/MISSING_DEPENDENCY/TYPE_MISMATCH/
    API_MISMATCH/OTHER) plus the strict and lenient compilation rates for a
    `compute_java_compilation_report` result."""
    counts = report["counts"]
    print("=" * 56)
    print(title)
    print("=" * 56)
    print()
    for category in ("PASS", "SYNTAX_ERROR", "MISSING_DEPENDENCY", "TYPE_MISMATCH",
                     "API_MISMATCH", "OTHER"):
        print(f"{category}: {counts.get(category, 0)}")
        print()
    print(f"Strict Compilation Rate: {report['strict_rate'] * 100:.2f}%")
    print()
    print(f"Lenient Compilation Rate: {report['lenient_rate'] * 100:.2f}%")
    print("=" * 56)

## Ground Truth Validation

In [17]:

GT_JAVA_CATEGORY_DISPLAY = {"TYPE_MISMATCH": "TYPE_ERROR"}


def _display_java_category(raw_category: str) -> str:
    """Map the shared classify_javac_stderr() category name to the Ground Truth Validation
    taxonomy requested for this stage (TYPE_MISMATCH -> TYPE_ERROR); every other category name
    is already shared verbatim (PASS/SYNTAX_ERROR/MISSING_DEPENDENCY/API_MISMATCH/OTHER)."""
    return GT_JAVA_CATEGORY_DISPLAY.get(raw_category, raw_category)


def is_lenient_valid_reference(category: str) -> bool:
    """Ground-truth-specific lenient rule: a REFERENCE program is rejected ONLY when it has a
    real SYNTAX_ERROR. Missing symbols/packages/imports/external dependencies (and any other
    non-syntax compiler complaint) are expected noise from compiling an isolated snippet
    outside of its original enclosing project, not evidence the reference itself is wrong."""
    return category != "SYNTAX_ERROR"


_JAVALANG_READY: Optional[bool] = None


def _ensure_javalang() -> bool:
    """Best-effort import of `javalang` (a pure-Python, JVM-free Java parser) used as an
    independent 'does a real Java parser accept this code' check, separate from javac itself.
    Cached after the first attempt; never raises - returns False if the package is missing."""
    global _JAVALANG_READY
    if _JAVALANG_READY is not None:
        return _JAVALANG_READY
    try:
        import javalang  # noqa: F401
        _JAVALANG_READY = True
    except Exception:
        logger.warning(
            "`javalang` not available - skipping the standalone Java-parser check "
            "(javac's own syntax diagnostics during compilation still apply).")
        _JAVALANG_READY = False
    return _JAVALANG_READY


def parse_java_with_javalang(code: str) -> "tuple[Optional[bool], str]":
    """Try to parse `code` with `javalang` (wrapping bare method-only snippets in a temporary
    `GeneratedProgram` class first, exactly like the compiler path). Returns `(True, "")` if
    it parses, `(False, error_message)` on a real syntax error, or `(None, "")` if `javalang`
    itself is unavailable (caller should then rely solely on the javac diagnostics captured
    later)."""
    if not _ensure_javalang():
        return None, ""
    import javalang
    try:
        source, _ = _prepare_java_source(code)
        javalang.parse.parse(source)
        return True, ""
    except javalang.parser.JavaSyntaxError as exc:
        return False, f"javalang syntax error: {exc}"
    except Exception as exc:      
        logger.warning("javalang parse raised a non-syntax exception (%s); "
                       "deferring to javac.", exc)
        return None, ""


def validate_java_reference(sample_id, code: str, timeout: int = 20) -> Dict[str, object]:
    """Validate one Java ground-truth reference: emptiness -> javalang parser pre-check ->
    javac compile (via the existing `analyze_java_compilation`) -> classification. Returns a
    flat dict matching the `ground_truth_validation.csv` schema plus the two internal
    strict/lenient flags used for filtering."""
    code = code or ""
    if not code.strip():
        return {
            "sample_id": sample_id, "language": "java", "status": "INVALID",
            "category": "OTHER", "compiler_message": "Empty source code.",
            "used_in_evaluation": False, "strict_valid": False, "lenient_valid": False,
        }

    parser_ok, parser_message = parse_java_with_javalang(code)
    if parser_ok is False:
        return {
            "sample_id": sample_id, "language": "java", "status": "INVALID",
            "category": "SYNTAX_ERROR", "compiler_message": parser_message,
            "used_in_evaluation": False, "strict_valid": False, "lenient_valid": False,
        }

    analysis = analyze_java_compilation(code, timeout=timeout)
    category = _display_java_category(analysis["category"])
    lenient_valid = is_lenient_valid_reference(category)
    return {
        "sample_id": sample_id, "language": "java",
        "status": "VALID" if lenient_valid else "INVALID",
        "category": category, "compiler_message": (analysis["compiler_output"] or "")[:1000],
        "used_in_evaluation": lenient_valid,
        "strict_valid": analysis["is_strict_compile_success"], "lenient_valid": lenient_valid,
    }


CSHARP_SYNTAX_ERROR_CODES = (
    "CS1519", "CS1520", "CS1002", "CS1513", "CS1022", "CS1026", "CS1003", "CS1525", "CS8025",
)
CSHARP_TYPE_ERROR_CODES = ("CS0029", "CS1503", "CS0266", "CS0019")
CSHARP_API_MISMATCH_CODES = ("CS1501", "CS1061", "CS0117", "CS7036")
CSHARP_MISSING_DEPENDENCY_CODES = ("CS0246", "CS0234", "CS0103")


def _csharp_codes_present(codes: "tuple[str, ...]", text: str) -> bool:
    """True if any of `codes` (e.g. 'CS0246') appears as a whole word in `text`."""
    return any(re.search(rf"\b{code}\b", text) for code in codes)


def classify_csharp_diagnostics(stderr: str) -> str:
    """Classify a `dotnet build`/`csc` failure into PASS/SYNTAX_ERROR/MISSING_DEPENDENCY/
    TYPE_ERROR/API_MISMATCH/OTHER via its `CS####` diagnostic codes, checked in the same
    severity order as `classify_javac_stderr`: a real syntax error always dominates.

    NOTE (Roslyn): `dotnet build`/`csc` ARE the Roslyn C# compiler - parsing is the first
    phase of every Roslyn compilation, so a syntax-only failure surfaces here as one of the
    `CSHARP_SYNTAX_ERROR_CODES`. A best-effort, fully standalone Roslyn *parse-only* pre-check
    (independent of a full compile) is additionally attempted in `parse_csharp_with_roslyn`
    below whenever a `Microsoft.CodeAnalysis.CSharp` host can be built.
    """
    text = stderr or ""
    if not text.strip():
        return "OTHER"
    if _csharp_codes_present(CSHARP_SYNTAX_ERROR_CODES, text) or \
       re.search(r"error CS\d+.*(?:expected|invalid token|unexpected)", text, re.IGNORECASE):
        return "SYNTAX_ERROR"
    if _csharp_codes_present(CSHARP_TYPE_ERROR_CODES, text):
        return "TYPE_ERROR"
    if _csharp_codes_present(CSHARP_API_MISMATCH_CODES, text):
        return "API_MISMATCH"
    if _csharp_codes_present(CSHARP_MISSING_DEPENDENCY_CODES, text):
        return "MISSING_DEPENDENCY"
    return "OTHER"


_ROSLYN_PARSER_EXE: Optional[str] = None
_ROSLYN_SETUP_ATTEMPTED = False


def _ensure_roslyn_parser(timeout: int = 180) -> Optional[str]:
    """Best-effort, ONE-TIME build of a tiny standalone Roslyn syntax-checker console app
    (references `Microsoft.CodeAnalysis.CSharp`, reads C# source on stdin, calls
    `CSharpSyntaxTree.ParseText(...).GetDiagnostics()`, and prints "OK" or each parse error on
    stdout). Requires `dotnet` + NuGet restore (network access) - if either is unavailable or
    the build fails/times out for any reason, this permanently caches `None` for the rest of
    the run and callers fall back to the compiler's own syntax diagnostics
    (`classify_csharp_diagnostics`) instead of raising."""
    global _ROSLYN_PARSER_EXE, _ROSLYN_SETUP_ATTEMPTED
    if _ROSLYN_SETUP_ATTEMPTED:
        return _ROSLYN_PARSER_EXE
    _ROSLYN_SETUP_ATTEMPTED = True
    dotnet = _ensure_dotnet()
    if not dotnet or Path(dotnet).name != "dotnet":
        logger.warning(
            "dotnet CLI not available - skipping the standalone Roslyn parse pre-check "
            "(the compiler's own syntax diagnostics still apply via classify_csharp_diagnostics).")
        return None
    try:
        proj_dir = Path(tempfile.gettempdir()) / "roslyn_parser_host"
        if not any(proj_dir.rglob("roslyn_parser_host.dll")):
            proj_dir.mkdir(parents=True, exist_ok=True)
            (proj_dir / "Program.cs").write_text(
                "using System;\n"
                "using System.Linq;\n"
                "using Microsoft.CodeAnalysis;\n"
                "using Microsoft.CodeAnalysis.CSharp;\n"
                "var source = Console.In.ReadToEnd();\n"
                "var tree = CSharpSyntaxTree.ParseText(source);\n"
                "var errors = tree.GetDiagnostics()\n"
                "    .Where(d => d.Severity == DiagnosticSeverity.Error)\n"
                "    .ToList();\n"
                "if (errors.Count == 0) { Console.WriteLine(\"OK\"); }\n"
                "else { foreach (var e in errors) Console.WriteLine(e.ToString()); }\n",
                encoding="utf-8",
            )
            (proj_dir / "roslyn_parser_host.csproj").write_text(
                '<Project Sdk="Microsoft.NET.Sdk">'
                "<PropertyGroup><OutputType>Exe</OutputType>"
                "<TargetFramework>net8.0</TargetFramework>"
                "<ImplicitUsings>enable</ImplicitUsings></PropertyGroup>"
                "<ItemGroup>"
                '<PackageReference Include="Microsoft.CodeAnalysis.CSharp" Version="4.9.2" />'
                "</ItemGroup></Project>",
                encoding="utf-8",
            )
            result = subprocess.run(
                [dotnet, "build", str(proj_dir), "-c",
                 "Release", "-v", "quiet"],
                capture_output=True, text=True, timeout=timeout,
            )
            if result.returncode != 0:
                logger.warning(
                    "Roslyn parser host build failed (likely no NuGet/network access) - "
                    "skipping the standalone Roslyn parse pre-check.\n%s",
                    (result.stderr or "")[:500])
                return None
        found_dll = next(proj_dir.rglob("roslyn_parser_host.dll"), None)
        if not found_dll:
            return None
        _ROSLYN_PARSER_EXE = str(found_dll)
        logger.info("Roslyn parser host built at %s", _ROSLYN_PARSER_EXE)
        return _ROSLYN_PARSER_EXE
    except Exception:
        logger.exception(
            "Roslyn parser host setup failed - skipping the standalone Roslyn parse pre-check.")
        return None


def parse_csharp_with_roslyn(code: str, timeout: int = 15) -> "tuple[Optional[bool], str]":
    """Run the one-time-built Roslyn syntax-checker host on `code` (wrapping bare-method
    snippets first, exactly like the compiler path). Returns `(True, "")` if Roslyn's own
    parser reports zero errors, `(False, diagnostics)` on a real parse error, or
    `(None, "")` if the standalone Roslyn host could not be built/run (caller should then rely
    on `classify_csharp_diagnostics` from the full compile step instead)."""
    dll = _ensure_roslyn_parser()
    if not dll:
        return None, ""
    dotnet = _ensure_dotnet()
    try:
        source = _prepare_csharp_source(code)
        result = subprocess.run(
            [dotnet, dll], input=source, capture_output=True, text=True, timeout=timeout,
        )
        output = (result.stdout or "").strip()
        if output == "OK":
            return True, ""
        return False, output[:1000]
    except Exception as exc:
        logger.warning(
            "Roslyn parse-only check raised %s; deferring to the compiler.", exc)
        return None, ""


def validate_csharp_reference(sample_id, code: str, timeout: int = 30) -> Dict[str, object]:
    """Validate one C# ground-truth reference: emptiness -> best-effort standalone Roslyn
    parse pre-check -> full compile via the existing `compile_csharp` -> classification via
    `classify_csharp_diagnostics`, using the same reference-specific lenient rule as Java
    (`is_lenient_valid_reference`: rejected only for a real SYNTAX_ERROR)."""
    code = code or ""
    if not code.strip():
        return {
            "sample_id": sample_id, "language": "csharp", "status": "INVALID",
            "category": "OTHER", "compiler_message": "Empty source code.",
            "used_in_evaluation": False, "strict_valid": False, "lenient_valid": False,
        }

    parser_ok, parser_message = parse_csharp_with_roslyn(code)
    if parser_ok is False:
        return {
            "sample_id": sample_id, "language": "csharp", "status": "INVALID",
            "category": "SYNTAX_ERROR", "compiler_message": parser_message,
            "used_in_evaluation": False, "strict_valid": False, "lenient_valid": False,
        }

    result = compile_csharp(code, timeout=timeout)
    is_success = bool(result.get("success"))
    diagnostics = result.get("stderr") or result.get("error") or ""
    category = "PASS" if is_success else classify_csharp_diagnostics(
        diagnostics)
    lenient_valid = is_success or is_lenient_valid_reference(category)
    return {
        "sample_id": sample_id, "language": "csharp",
        "status": "VALID" if lenient_valid else "INVALID",
        "category": category, "compiler_message": diagnostics[:1000],
        "used_in_evaluation": lenient_valid,
        "strict_valid": is_success, "lenient_valid": lenient_valid,
    }

In [18]:
GT_DISPLAY_CATEGORIES = ("PASS", "SYNTAX_ERROR", "MISSING_DEPENDENCY", "TYPE_ERROR",
                         "API_MISMATCH", "OTHER")


def summarize_java_reference_validation(results: List[Dict[str, object]]) -> Dict[str, object]:
    """Aggregate `validate_java_reference` results into total/strict-valid/lenient-valid/
    category counts, and print the requested Java summary block."""
    total = len(results)
    strict_valid = sum(1 for r in results if r["strict_valid"])
    lenient_valid = sum(1 for r in results if r["lenient_valid"])
    counts = Counter(r["category"] for r in results)
    other_total = counts.get("TYPE_ERROR", 0) + \
        counts.get("API_MISMATCH", 0) + counts.get("OTHER", 0)

    print("Java References")
    print()
    print(f"Total: {total}")
    print()
    print(f"Strict Valid:\n{strict_valid}")
    print()
    print(f"Lenient Valid:\n{lenient_valid}")
    print()
    print(f"Syntax Errors:\n{counts.get('SYNTAX_ERROR', 0)}")
    print()
    print(f"Missing Dependencies:\n{counts.get('MISSING_DEPENDENCY', 0)}")
    print()
    print(f"Other Errors:\n{other_total}")
    print()
    print("Category breakdown:")
    for category in GT_DISPLAY_CATEGORIES:
        print(f"  {category}: {counts.get(category, 0)}")
    print()
    return {
        "total": total, "strict_valid": strict_valid, "lenient_valid": lenient_valid,
        "counts": dict(counts),
        "strict_rate": strict_valid / max(1, total),
        "lenient_rate": lenient_valid / max(1, total),
    }


def summarize_csharp_reference_validation(results: List[Dict[str, object]]) -> Dict[str, object]:
    """Aggregate `validate_csharp_reference` results into total/valid/invalid/category counts,
    and print the requested C# summary block."""
    total = len(results)
    valid = sum(1 for r in results if r["lenient_valid"])
    invalid = total - valid
    strict_valid = sum(1 for r in results if r["strict_valid"])
    counts = Counter(r["category"] for r in results)

    print("C# References")
    print()
    print(f"Total: {total}")
    print()
    print(f"Valid:\n{valid}")
    print()
    print(f"Invalid:\n{invalid}")
    print()
    print("Category breakdown:")
    for category in GT_DISPLAY_CATEGORIES:
        print(f"  {category}: {counts.get(category, 0)}")
    print()
    return {
        "total": total, "valid": valid, "invalid": invalid, "counts": dict(counts),
        "strict_rate": strict_valid / max(1, total),
        "lenient_rate": valid / max(1, total),
    }


print("=" * 55)
print("Ground Truth Validation")
print("=" * 55)

java_gt_source = list(test_subset[STAGE1_CODE_COL])
csharp_gt_source = [p["cs"] for p in eval_pairs]

logger.info("Validating %d Java ground-truth references ...",
            len(java_gt_source))
java_validation_results = [validate_java_reference(
    i, code) for i, code in enumerate(java_gt_source)]

logger.info("Validating %d C# ground-truth references ...",
            len(csharp_gt_source))
csharp_validation_results = [validate_csharp_reference(
    i, code) for i, code in enumerate(csharp_gt_source)]

print()
print("=" * 55)
print("Ground Truth Validation Summary")
print("=" * 55)
print()
java_gt_stats = summarize_java_reference_validation(java_validation_results)
csharp_gt_stats = summarize_csharp_reference_validation(
    csharp_validation_results)

GROUND_TRUTH_CSV_PATH = "ground_truth_validation.csv"
_GT_CSV_FIELDS = ["sample_id", "language", "status",
                  "category", "compiler_message", "used_in_evaluation"]
with open(GROUND_TRUTH_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=_GT_CSV_FIELDS)
    writer.writeheader()
    for row in (java_validation_results + csharp_validation_results):
        writer.writerow({k: row[k] for k in _GT_CSV_FIELDS})
logger.info("Saved ground truth validation results to '%s' (%d rows).",
            GROUND_TRUTH_CSV_PATH, len(java_validation_results) + len(csharp_validation_results))

java_valid_indices = [i for i, r in enumerate(
    java_validation_results) if r["used_in_evaluation"]]
csharp_valid_indices = [i for i, r in enumerate(
    csharp_validation_results) if r["used_in_evaluation"]]

stage1_gt_excluded_count = len(test_subset) - len(java_valid_indices)
stage2_gt_excluded_count = len(eval_pairs) - len(csharp_valid_indices)

test_subset = test_subset.select(java_valid_indices)
eval_pairs = [eval_pairs[i] for i in csharp_valid_indices]

if len(test_subset) == 0:
    raise ValueError(
        "All Stage 1 Java references were rejected by Ground Truth Validation - check "
        "javac/toolchain availability (see the logged javac diagnostics above) before "
        "proceeding.")
if len(eval_pairs) == 0:
    raise ValueError(
        "All Stage 2 C# references were rejected by Ground Truth Validation - check "
        "dotnet/toolchain availability (see the logged compiler diagnostics above) before "
        "proceeding.")

print("=" * 55)
print("Ground Truth Samples Evaluated:")
print(len(test_subset) + len(eval_pairs))
print("Ground Truth Samples Excluded:")
print(stage1_gt_excluded_count + stage2_gt_excluded_count)
print("=" * 55)
print(
    f"  Java -> evaluated={len(test_subset)}, excluded={stage1_gt_excluded_count}")
print(
    f"  C#   -> evaluated={len(eval_pairs)}, excluded={stage2_gt_excluded_count}")

ground_truth_feasibility = {
    "stage1_java": {"strict": java_gt_stats["strict_rate"], "lenient": java_gt_stats["lenient_rate"]},
    "stage2_csharp": {"strict": csharp_gt_stats["strict_rate"], "lenient": csharp_gt_stats["lenient_rate"]},
}
ground_truth_validation_report = {
    "java": java_gt_stats, "csharp": csharp_gt_stats,
    "stage1_excluded": stage1_gt_excluded_count, "stage2_excluded": stage2_gt_excluded_count,
}

03:48:46 | INFO | EvalOnlyNotebook | Validating 100 Java ground-truth references ...


Ground Truth Validation


03:49:45 | INFO | EvalOnlyNotebook | Validating 50 C# ground-truth references ...
03:49:45 | INFO | EvalOnlyNotebook | dotnet not found - attempting Microsoft's official install script ...
03:50:06 | INFO | EvalOnlyNotebook | Roslyn parser host built at /tmp/roslyn_parser_host/bin/Release/net8.0/roslyn_parser_host.dll
03:51:44 | INFO | EvalOnlyNotebook | Saved ground truth validation results to 'ground_truth_validation.csv' (150 rows).



Ground Truth Validation Summary

Java References

Total: 100

Strict Valid:
0

Lenient Valid:
100

Syntax Errors:
0

Missing Dependencies:
99

Other Errors:
1

Category breakdown:
  PASS: 0
  SYNTAX_ERROR: 0
  MISSING_DEPENDENCY: 99
  TYPE_ERROR: 0
  API_MISMATCH: 1
  OTHER: 0

C# References

Total: 50

Valid:
50

Invalid:
0

Category breakdown:
  PASS: 0
  SYNTAX_ERROR: 0
  MISSING_DEPENDENCY: 0
  TYPE_ERROR: 0
  API_MISMATCH: 0
  OTHER: 50

Ground Truth Samples Evaluated:
150
Ground Truth Samples Excluded:
0
  Java -> evaluated=100, excluded=0
  C#   -> evaluated=50, excluded=0


## Stage 1 Evaluation (NL → Java)

In [19]:
STAGE1_RESPONSE_MARKER = "### Java Solution:"


def add_java_hint(instruction: str) -> str:
    """Append a light hint nudging the model to produce a full class, not just a bare method."""
    return (f"{instruction.strip()}\n\n"
            "Provide a complete, compilable Java solution (including the class definition).")


def clean_instruction(doc: str) -> str:
    """Normalize whitespace in a raw docstring/instruction."""
    return re.sub(r"\s+", " ", doc or "").strip()


def build_stage1_prompt(instruction: str) -> str:
    """Build the exact NL->Java inference prompt used during Stage 1 fine-tuning."""
    instruction = add_java_hint(clean_instruction(instruction))
    return (
        "You are an expert Java programmer. Write Java code that solves the following "
        f"problem.\n\n### Instruction:\n{instruction}\n\n{STAGE1_RESPONSE_MARKER}\n"
    )

In [20]:
def _run_generation(gen_model, gen_tokenizer, prompt: str, max_new_tokens: int,
                    response_marker: Optional[str] = None) -> str:
    """Run greedy generation for a single prompt and strip the prompt/marker from the output."""
    FastLanguageModel.for_inference(gen_model)
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True,
                           max_length=CFG.MAX_LENGTH).to(gen_model.device)
    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=gen_tokenizer.pad_token_id or gen_tokenizer.eos_token_id,
        )
    full_text = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    if response_marker and response_marker in full_text:
        generated = full_text.split(response_marker, 1)[1]
    else:
        prompt_text = gen_tokenizer.decode(
            inputs["input_ids"][0], skip_special_tokens=True)
        generated = full_text[len(prompt_text):] if full_text.startswith(
            prompt_text) else full_text
    return generated.strip()


def generate_java_from_nl(gen_model, gen_tokenizer, instruction: str,
                          max_new_tokens: int = 300) -> str:
    """Generate a Java solution string for a single NL instruction."""
    prompt = build_stage1_prompt(instruction)
    return _run_generation(gen_model, gen_tokenizer, prompt, max_new_tokens,
                           response_marker=STAGE1_RESPONSE_MARKER)


def generate_predictions(gen_model, gen_tokenizer, dataset, doc_col: str = STAGE1_DOC_COL,
                         max_new_tokens: int = 300, log_every: int = 20) -> List[str]:
    """Generate Stage 1 predictions for every example in `dataset`."""
    preds = []
    n = len(dataset)
    start = time.time()
    for i, doc in enumerate(dataset[doc_col]):
        preds.append(generate_java_from_nl(
            gen_model, gen_tokenizer, doc, max_new_tokens))
        if (i + 1) % log_every == 0 or (i + 1) == n:
            elapsed = time.time() - start
            logger.info("Generated %d/%d Stage 1 predictions (%.1fs elapsed).",
                        i + 1, n, elapsed)
    return preds

In [21]:
def compute_code_metrics(predictions: List[str], references: List[str]) -> Dict[str, float]:
    """Compute BLEU, CodeBLEU (+ sub-scores, and a no-dataflow variant), and CodeBERTScore."""
    import sacrebleu
    from codebleu import calc_codebleu

    report: Dict[str, float] = {}

    # --- BLEU (added here for parity with Stage 2's evaluate_csharp, which already reports it)
    bleu = sacrebleu.corpus_bleu(predictions, [references])
    report["BLEU"] = bleu.score

    # --- CodeBLEU (standard weights) + its 4 sub-components ---
    codebleu_result = calc_codebleu(references, predictions, lang="java")
    report["CodeBLEU"] = codebleu_result["codebleu"]
    report["ngram_match_score"] = codebleu_result["ngram_match_score"]
    report["weighted_ngram_match_score"] = codebleu_result["weighted_ngram_match_score"]
    report["syntax_match_score"] = codebleu_result["syntax_match_score"]
    report["dataflow_match_score"] = codebleu_result["dataflow_match_score"]

    codebleu_no_dataflow = calc_codebleu(
        references, predictions, lang="java", weights=(1 / 3, 1 / 3, 1 / 3, 0.0))
    report["CodeBLEU_no_dataflow"] = codebleu_no_dataflow["codebleu"]

    try:
        from code_bert_score import score as code_bert_score
        result = code_bert_score(predictions, references, lang="java")
        f1 = result[2]
        report["CodeBERTScore_F1"] = float(f1.mean())
    except Exception:
        logger.exception(
            "CodeBERTScore computation failed; skipping this metric.")
        report["CodeBERTScore_F1"] = float("nan")

    return report

In [22]:
print("Ground Truth Samples Evaluated:")
print(len(test_subset))
print("Ground Truth Samples Excluded:")
print(stage1_gt_excluded_count)

references_stage1 = list(test_subset[STAGE1_CODE_COL])

logger.info("Generating Stage 1 predictions with the FINE-TUNED model ...")
finetuned_stage1_predictions = generate_predictions(
    stage1_model, stage1_tokenizer, test_subset)
finetuned_stage1_metrics = compute_code_metrics(
    finetuned_stage1_predictions, references_stage1)

logger.info("Generating Stage 1 predictions with the BASE model ...")
base_stage1_predictions = generate_predictions(
    base_model, base_tokenizer, test_subset)
base_stage1_metrics = compute_code_metrics(
    base_stage1_predictions, references_stage1)

stage1_evaluation = {
    "fine_tuned": finetuned_stage1_metrics, "base": base_stage1_metrics}
print("Stage 1 fine-tuned metrics:",
      json.dumps(finetuned_stage1_metrics, indent=2))
print("Stage 1 base metrics:", json.dumps(base_stage1_metrics, indent=2))

03:51:44 | INFO | EvalOnlyNotebook | Generating Stage 1 predictions with the FINE-TUNED model ...


Ground Truth Samples Evaluated:
100
Ground Truth Samples Excluded:
0


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

config.json:   0%|          | 0.00/696 [00:00<?, ?B/s]

03:56:45 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
03:56:45 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/tokenizer_config.json "HTTP/1.1 200 OK"
03:56:45 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/tokenizer_config.json "HTTP/1.1 200 OK"


...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

03:56:56 | INFO | EvalOnlyNotebook | Generating Stage 1 predictions with the BASE model ...
04:01:07 | INFO | EvalOnlyNotebook | Generated 20/100 Stage 1 predictions (250.9s elapsed).
04:05:25 | INFO | EvalOnlyNotebook | Generated 40/100 Stage 1 predictions (509.1s elapsed).
04:09:37 | INFO | EvalOnlyNotebook | Generated 60/100 Stage 1 predictions (760.2s elapsed).
04:13:43 | INFO | EvalOnlyNotebook | Generated 80/100 Stage 1 predictions (1006.6s elapsed).
04:17:50 | INFO | EvalOnlyNotebook | Generated 100/100 Stage 1 predictions (1253.8s elapsed).
04:17:52 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04:17:52 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/config.json "HTTP/1.1 200 OK"
04:17:52 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/main/tokenizer_

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

04:17:52 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/commits/main "HTTP/1.1 200 OK"
04:17:53 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/discussions?p=0 "HTTP/1.1 200 OK"
04:17:53 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
04:17:53 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"
04:17:53 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/refs%2Fpr%2F1/model.safetensors "HTTP/1.1 302 Found"


Stage 1 fine-tuned metrics: {
  "BLEU": 30.710513732756453,
  "CodeBLEU": 0.3350421594619504,
  "ngram_match_score": 0.18652433719325776,
  "weighted_ngram_match_score": 0.20896116715181048,
  "syntax_match_score": 0.287598944591029,
  "dataflow_match_score": 0.6570841889117043,
  "CodeBLEU_no_dataflow": 0.2276948163120324,
  "CodeBERTScore_F1": 0.785347580909729
}
Stage 1 base metrics: {
  "BLEU": 4.529076479656573,
  "CodeBLEU": 0.3022941544224457,
  "ngram_match_score": 0.011888441344443121,
  "weighted_ngram_match_score": 0.08024836614076543,
  "syntax_match_score": 0.2556435063031369,
  "dataflow_match_score": 0.8613963039014374,
  "CodeBLEU_no_dataflow": 0.11592677126278182,
  "CodeBERTScore_F1": 0.6709613800048828
}


## Stage 2 Evaluation (Java → C#)

In [23]:
STAGE2_INSTRUCTION = (
    "Translate the following Java code to C#. Preserve behavior and structure exactly."
)

STAGE2_INFERENCE_GUARDRAILS = (
    "Rules:\n"
    "1. Output ONLY the translated C# code, no explanations.\n"
    "2. Do not include markdown code fences.\n"
    "3. Preserve method/class names where possible.\n"
    "4. Use idiomatic C# equivalents for Java-specific constructs "
    "(e.g. System.out.println -> Console.WriteLine).\n"
)


def build_stage2_strict_inference_prompt(java_code: str) -> str:
    """Build the strict guard-railed Java->C# translation prompt used in fine-tuning."""
    return (
        f"{STAGE2_INSTRUCTION}\n\n{STAGE2_INFERENCE_GUARDRAILS}\n"
        f"### Java:\n{java_code.strip()}\n\n### C#:\n"
    )


def strip_unwanted_csharp_modifiers(code: str) -> str:
    """Remove markdown fences and normalize whitespace in a raw C# generation."""
    code = re.sub(r"```(?:csharp|cs|c#)?\s*", "", code, flags=re.IGNORECASE)
    code = code.replace("```", "")
    return code.strip()


def _generate_csharp(gen_model, gen_tokenizer, java_code: str,
                     max_new_tokens: int = 400) -> str:
    """Generate a single Java->C# translation and clean the raw output."""
    prompt = build_stage2_strict_inference_prompt(java_code)
    raw = _run_generation(gen_model, gen_tokenizer, prompt, max_new_tokens,
                          response_marker="### C#:")
    return strip_unwanted_csharp_modifiers(raw)

In [24]:
def _run_stage2_eval_loop(gen_model, gen_tokenizer, pairs: List[Dict[str, str]],
                          max_new_tokens: int = 400, log_every: int = 10) -> List[str]:
    """Run Java->C# generation over every pair, returning the list of predictions."""
    preds = []
    n = len(pairs)
    start = time.time()
    for i, pair in enumerate(pairs):
        preds.append(_generate_csharp(
            gen_model, gen_tokenizer, pair["java"], max_new_tokens))
        if (i + 1) % log_every == 0 or (i + 1) == n:
            elapsed = time.time() - start
            logger.info("Generated %d/%d Stage 2 predictions (%.1fs elapsed).",
                        i + 1, n, elapsed)
    return preds

In [25]:
def _get_csharp_ts_parser():
    """Build a tree-sitter parser for C# (lazily, so a missing grammar doesn't break imports)."""
    import tree_sitter_c_sharp as tscsharp
    from tree_sitter import Language, Parser
    language = Language(tscsharp.language())
    parser = Parser(language)
    return parser


def _ast_node_types(parser, code: str) -> Counter:
    """Parse `code` and return a bag-of-words Counter over AST node types."""
    tree = parser.parse(bytes(code, "utf-8"))
    counts: Counter = Counter()

    def _walk(node):
        counts[node.type] += 1
        for child in node.children:
            _walk(child)

    _walk(tree.root_node)
    return counts


def _cosine_similarity(a: Counter, b: Counter) -> float:
    """Cosine similarity between two node-type bag-of-words Counters."""
    if not a or not b:
        return 0.0
    keys = set(a) | set(b)
    dot = sum(a.get(k, 0) * b.get(k, 0) for k in keys)
    norm_a = math.sqrt(sum(v * v for v in a.values()))
    norm_b = math.sqrt(sum(v * v for v in b.values()))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def is_syntactically_valid_csharp_ts(code: str) -> bool:
    """Best-effort tree-sitter syntax check (no ERROR nodes in the parse tree)."""
    try:
        parser = _get_csharp_ts_parser()
        tree = parser.parse(bytes(code, "utf-8"))
        return "ERROR" not in str(tree.root_node) if tree.root_node.has_error else True
    except Exception:
        return False


def evaluate_csharp(predictions: List[str], references: List[str]) -> Dict[str, float]:
    """Compute BLEU, CodeBLEU, Exact Match, Syntax Accuracy, and AST Similarity for C#.

    NOTE: this deliberately does NOT compute a compilation rate here - the reference
    notebook's `evaluate_csharp` had its own internal mono/tree-sitter-based compilation
    check, which duplicated the more complete `compile_csharp` real-compiler suite (Section
    11). That duplication was removed; use `compute_compilation_rate`/
    `compute_lenient_compilation_rate` for the authoritative C# compilation metrics.
    """
    import sacrebleu
    from codebleu import calc_codebleu

    report: Dict[str, float] = {}

    bleu = sacrebleu.corpus_bleu(predictions, [references])
    report["BLEU"] = bleu.score

    codebleu_result = calc_codebleu(references, predictions, lang="c_sharp")
    report["CodeBLEU"] = codebleu_result["codebleu"]

    exact_matches = sum(
        1 for p, r in zip(predictions, references) if p.strip() == r.strip())
    report["ExactMatch"] = exact_matches / max(1, len(predictions))

    try:
        valid_count = sum(
            1 for p in predictions if is_syntactically_valid_csharp_ts(p))
        report["SyntaxAccuracy"] = valid_count / max(1, len(predictions))
    except Exception:
        logger.exception(
            "C# syntax-validity check failed; skipping SyntaxAccuracy.")
        report["SyntaxAccuracy"] = float("nan")

    try:
        parser = _get_csharp_ts_parser()
        sims = []
        for p, r in zip(predictions, references):
            try:
                sims.append(_cosine_similarity(_ast_node_types(parser, p),
                                               _ast_node_types(parser, r)))
            except Exception:
                sims.append(0.0)
        report["AST_Similarity"] = float(
            np.mean(sims)) if sims else float("nan")
    except Exception:
        logger.exception(
            "AST similarity computation failed; skipping AST_Similarity.")
        report["AST_Similarity"] = float("nan")

    return report

In [33]:
print("Ground Truth Samples Evaluated:")
print(len(eval_pairs))
print("Ground Truth Samples Excluded:")
print(stage2_gt_excluded_count)

references_stage2 = [p["cs"] for p in eval_pairs]

logger.info("Generating Stage 2 predictions with the FINE-TUNED model ...")
finetuned_stage2_predictions = _run_stage2_eval_loop(
    stage2_model, stage2_tokenizer, eval_pairs)
stage2_report = evaluate_csharp(
    finetuned_stage2_predictions, references_stage2)

logger.info("Generating Stage 2 predictions with the BASE model ...")
base_stage2_predictions = _run_stage2_eval_loop(
    base_model, base_tokenizer, eval_pairs)
stage2_base_report = evaluate_csharp(
    base_stage2_predictions, references_stage2)

stage2_evaluation = {"fine_tuned": stage2_report, "base": stage2_base_report}

05:04:27 | INFO | EvalOnlyNotebook | Generating Stage 2 predictions with the FINE-TUNED model ...


Ground Truth Samples Evaluated:
50
Ground Truth Samples Excluded:
0


05:04:44 | INFO | EvalOnlyNotebook | Generated 10/50 Stage 2 predictions (17.1s elapsed).
05:05:09 | INFO | EvalOnlyNotebook | Generated 20/50 Stage 2 predictions (41.4s elapsed).
05:05:28 | INFO | EvalOnlyNotebook | Generated 30/50 Stage 2 predictions (60.9s elapsed).
05:05:47 | INFO | EvalOnlyNotebook | Generated 40/50 Stage 2 predictions (79.3s elapsed).
05:06:10 | INFO | EvalOnlyNotebook | Generated 50/50 Stage 2 predictions (102.8s elapsed).
05:06:10 | INFO | EvalOnlyNotebook | Generating Stage 2 predictions with the BASE model ...
05:08:23 | INFO | EvalOnlyNotebook | Generated 10/50 Stage 2 predictions (133.0s elapsed).
05:10:46 | INFO | EvalOnlyNotebook | Generated 20/50 Stage 2 predictions (276.2s elapsed).
05:13:13 | INFO | EvalOnlyNotebook | Generated 30/50 Stage 2 predictions (422.8s elapsed).
05:15:42 | INFO | EvalOnlyNotebook | Generated 40/50 Stage 2 predictions (572.2s elapsed).
05:17:59 | INFO | EvalOnlyNotebook | Generated 50/50 Stage 2 predictions (709.3s elapsed).


In [ ]:
finetuned_java_report = compute_java_compilation_report(
    finetuned_stage1_predictions)
base_java_report = compute_java_compilation_report(base_stage1_predictions)

stage1_evaluation["compilation"] = {
    "fine_tuned": {
        "JavaCompilationRate": finetuned_java_report["strict_rate"],
        "JavaLenientCompilationRate": finetuned_java_report["lenient_rate"],
    },
    "base": {
        "JavaCompilationRate": base_java_report["strict_rate"],
        "JavaLenientCompilationRate": base_java_report["lenient_rate"],
    },
    "ground_truth_ceiling": ground_truth_feasibility["stage1_java"],
    "category_counts": {
        "fine_tuned": dict(finetuned_java_report["counts"]),
        "base": dict(base_java_report["counts"]),
    },
}
print("Stage 1 compilation rates:", json.dumps(
    stage1_evaluation["compilation"], indent=2))

print_compilation_summary(finetuned_java_report,
                          title="Compilation Summary - Java (fine-tuned)")
print_compilation_summary(base_java_report,
                          title="Compilation Summary - Java (base)")

diagnose_compilation_rate(
    finetuned_stage1_predictions, _prepare_java_source, compile_java,
    "Java (fine-tuned)", stage1_evaluation["compilation"]["fine_tuned"]["JavaCompilationRate"])
diagnose_compilation_rate(
    base_stage1_predictions, _prepare_java_source, compile_java,
    "Java (base)", stage1_evaluation["compilation"]["base"]["JavaCompilationRate"])

In [34]:
stage2_evaluation["compilation"] = {
    "fine_tuned": {
        "CSharpCompilationRate": compute_compilation_rate(
            finetuned_stage2_predictions, "c_sharp"),
        "CSharpLenientCompilationRate": compute_lenient_compilation_rate(
            finetuned_stage2_predictions, "c_sharp"),
    },
    "base": {
        "CSharpCompilationRate": compute_compilation_rate(base_stage2_predictions, "c_sharp"),
        "CSharpLenientCompilationRate": compute_lenient_compilation_rate(
            base_stage2_predictions, "c_sharp"),
    },
    "ground_truth_ceiling": ground_truth_feasibility["stage2_csharp"],
}
print("Stage 2 compilation rates:", json.dumps(
    stage2_evaluation["compilation"], indent=2))

diagnose_compilation_rate(
    finetuned_stage2_predictions, _prepare_csharp_source, compile_csharp,
    "C# (fine-tuned)", stage2_evaluation["compilation"]["fine_tuned"]["CSharpCompilationRate"])
diagnose_compilation_rate(
    base_stage2_predictions, _prepare_csharp_source, compile_csharp,
    "C# (base)", stage2_evaluation["compilation"]["base"]["CSharpCompilationRate"])

Stage 2 compilation rates: {
  "fine_tuned": {
    "CSharpCompilationRate": 0.0,
    "CSharpLenientCompilationRate": 1.0
  },
  "base": {
    "CSharpCompilationRate": 0.0,
    "CSharpLenientCompilationRate": 0.54
  },
  "ground_truth_ceiling": {
    "strict": 0.0,
    "lenient": 1.0
  }
}
DIAGNOSTIC: class-structured C# (fine-tuned) output (first 2 predictions)
--- C# (fine-tuned) prediction: raw model output ---
public TopMarginRecord(RecordInputStream in1){field_1_margin = in1.ReadDouble();}
--- C# (fine-tuned) prediction: class-structured source sent to the compiler ---
using System;
using System.Collections.Generic;

public class GeneratedProgram
{
public TopMarginRecord(RecordInputStream in1){field_1_margin = in1.ReadDouble();}
}

--- C# (fine-tuned) prediction: compile result = FAILED ---


--- C# (fine-tuned) prediction: raw model output ---
public virtual Antlr4.Runtime.Misc.IntervalSet Complement(int minElement, int maxElement){return this.Complement(Antlr4.Runtime.Misc.Interv

## Final Comparison Tables

In [36]:
def _fmt(value: float) -> str:
    """Format a metric value as a percentage-style or plain float string."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "N/A"
    return f"{value * 100:.2f}" if 0.0 <= value <= 1.0 else f"{value:.2f}"


def _diff(base_value: float, ft_value: float) -> str:
    """Format the (fine-tuned - base) improvement for a metric, prefixed with a sign."""
    if any(v is None or (isinstance(v, float) and math.isnan(v))
           for v in (base_value, ft_value)):
        return "N/A"
    delta = ft_value - base_value
    scaled = delta * 100 if 0.0 <= base_value <= 1.0 and 0.0 <= ft_value <= 1.0 else delta
    sign = "+" if scaled >= 0 else ""
    return f"{sign}{scaled:.2f}"


def _codebleu_value(metrics: Dict[str, float]) -> float:
    return metrics.get("CodeBLEU", float("nan"))


def _codebleu_no_dataflow_value(metrics: Dict[str, float]) -> float:
    return metrics.get("CodeBLEU_no_dataflow", float("nan"))


def _print_comparison_table(title: str, base_metrics: Dict[str, float],
                            ft_metrics: Dict[str, float], rows: List["tuple[str, str]"]) -> None:
    """Print a Metric / Base / Fine-tuned / Improvement table for an arbitrary set of rows.

    `rows` is a list of (display_label, metrics_key) tuples.
    """
    print("=" * 70)
    print(title)
    print("=" * 70)
    print(f"{'Metric':<32}{'Base':>12}{'Fine-tuned':>14}{'Improvement':>14}")
    print("-" * 70)
    for display_label, key in rows:
        base_v = base_metrics.get(key, float("nan"))
        ft_v = ft_metrics.get(key, float("nan"))
        print(
            f"{display_label:<32}{_fmt(base_v):>12}{_fmt(ft_v):>14}{_diff(base_v, ft_v):>14}")
    print("=" * 70)

In [38]:
stage1_base_all = {**stage1_evaluation["base"],
                   **stage1_evaluation["compilation"]["base"]}
stage1_ft_all = {**stage1_evaluation["fine_tuned"],
                 **stage1_evaluation["compilation"]["fine_tuned"]}

_print_comparison_table(
    "Stage 1 (NL -> Java)", stage1_base_all, stage1_ft_all,
    rows=[
        ("BLEU", "BLEU"),
        ("CodeBLEU", "CodeBLEU"),
        ("CodeBERTScore", "CodeBERTScore_F1"),
        ("ngram_match_score", "ngram_match_score"),
        ("weighted_ngram_match_score", "weighted_ngram_match_score"),
        ("syntax_match_score", "syntax_match_score"),
        ("dataflow_match_score", "dataflow_match_score"),
        ("CodeBLEU excl. dataflow", "CodeBLEU_no_dataflow"),
        ("Java Compilation Rate", "JavaCompilationRate"),
        ("Java Lenient Compilation Rate", "JavaLenientCompilationRate"),
    ],
)
print(f"(Ground truth Java compilation ceiling: strict="
      f"{ground_truth_feasibility['stage1_java']['strict'] * 100:.1f}%, lenient="
      f"{ground_truth_feasibility['stage1_java']['lenient'] * 100:.1f}%)\n")

stage2_base_all = {**stage2_evaluation["base"],
                   **stage2_evaluation["compilation"]["base"]}
stage2_ft_all = {**stage2_evaluation["fine_tuned"],
                 **stage2_evaluation["compilation"]["fine_tuned"]}

_print_comparison_table(
    "Stage 2 (Java -> C#)", stage2_base_all, stage2_ft_all,
    rows=[
        ("BLEU", "BLEU"),
        ("CodeBLEU", "CodeBLEU"),
        ("Exact Match", "ExactMatch"),
        ("Syntax Accuracy", "SyntaxAccuracy"),
        ("AST Similarity", "AST_Similarity"),
        ("C# Compilation Rate", "CSharpCompilationRate"),
        ("C# Lenient Compilation Rate", "CSharpLenientCompilationRate"),
    ],
)
print(f"(Ground truth C# compilation ceiling: strict="
      f"{ground_truth_feasibility['stage2_csharp']['strict'] * 100:.1f}%, lenient="
      f"{ground_truth_feasibility['stage2_csharp']['lenient'] * 100:.1f}%)")

Stage 1 (NL -> Java)
Metric                                  Base    Fine-tuned   Improvement
----------------------------------------------------------------------
BLEU                                    4.53         30.71        +26.18
CodeBLEU                               30.23         33.50         +3.27
CodeBERTScore                          67.10         78.53        +11.44
ngram_match_score                       1.19         18.65        +17.46
weighted_ngram_match_score              8.02         20.90        +12.87
syntax_match_score                     25.56         28.76         +3.20
dataflow_match_score                   86.14         65.71        -20.43
CodeBLEU excl. dataflow                11.59         22.77        +11.18
Java Compilation Rate                   0.00          0.00         +0.00
Java Lenient Compilation Rate          90.00         93.00         +3.00
(Ground truth Java compilation ceiling: strict=0.0%, lenient=100.0%)

Stage 2 (Java -> C#)
Metric        

## Overall Summary

In [39]:
print("=" * 70)
print("OVERALL EVALUATION SUMMARY")
print("=" * 70)

print("\n--- Stage 1: NL -> Java ---")
for label, metrics in (("Base", stage1_base_all), ("Fine-tuned", stage1_ft_all)):
    print(f"[{label}] BLEU={_fmt(metrics.get('BLEU'))}  "
          f"CodeBLEU={_fmt(metrics.get('CodeBLEU'))}  "
          f"CodeBERTScore={_fmt(metrics.get('CodeBERTScore_F1'))}  "
          f"JavaCompilation={_fmt(metrics.get('JavaCompilationRate'))}  "
          f"JavaLenientCompilation={_fmt(metrics.get('JavaLenientCompilationRate'))}")
print(f"Ground truth ceiling: strict="
      f"{ground_truth_feasibility['stage1_java']['strict'] * 100:.1f}%, lenient="
      f"{ground_truth_feasibility['stage1_java']['lenient'] * 100:.1f}%")

print("\n--- Stage 2: Java -> C# ---")
for label, metrics in (("Base", stage2_base_all), ("Fine-tuned", stage2_ft_all)):
    print(f"[{label}] BLEU={_fmt(metrics.get('BLEU'))}  "
          f"CodeBLEU={_fmt(metrics.get('CodeBLEU'))}  "
          f"AST_Similarity={_fmt(metrics.get('AST_Similarity'))}  "
          f"SyntaxAccuracy={_fmt(metrics.get('SyntaxAccuracy'))}  "
          f"CSharpCompilation={_fmt(metrics.get('CSharpCompilationRate'))}  "
          f"CSharpLenientCompilation={_fmt(metrics.get('CSharpLenientCompilationRate'))}")
print(f"Ground truth ceiling: strict="
      f"{ground_truth_feasibility['stage2_csharp']['strict'] * 100:.1f}%, lenient="
      f"{ground_truth_feasibility['stage2_csharp']['lenient'] * 100:.1f}%")

print("\n--- Ground Truth Validation ---")
print(f"Ground Truth Samples Evaluated: {len(test_subset) + len(eval_pairs)}")
print(f"Ground Truth Samples Excluded: "
      f"{stage1_gt_excluded_count + stage2_gt_excluded_count}")

print("=" * 70)

overall_evaluation_summary = {
    "stage1": stage1_evaluation,
    "stage2": stage2_evaluation,
    "ground_truth_feasibility": ground_truth_feasibility,
    "ground_truth_validation": ground_truth_validation_report,
}

OVERALL EVALUATION SUMMARY

--- Stage 1: NL -> Java ---
[Base] BLEU=4.53  CodeBLEU=30.23  CodeBERTScore=67.10  JavaCompilation=0.00  JavaLenientCompilation=90.00
[Fine-tuned] BLEU=30.71  CodeBLEU=33.50  CodeBERTScore=78.53  JavaCompilation=0.00  JavaLenientCompilation=93.00
Ground truth ceiling: strict=0.0%, lenient=100.0%

--- Stage 2: Java -> C# ---
[Base] BLEU=8.59  CodeBLEU=36.00  AST_Similarity=75.12  SyntaxAccuracy=2.00  CSharpCompilation=0.00  CSharpLenientCompilation=54.00
[Fine-tuned] BLEU=82.24  CodeBLEU=76.21  AST_Similarity=98.37  SyntaxAccuracy=92.00  CSharpCompilation=0.00  CSharpLenientCompilation=100.00
Ground truth ceiling: strict=0.0%, lenient=100.0%

--- Ground Truth Validation ---
Ground Truth Samples Evaluated: 150
Ground Truth Samples Excluded: 0
